# Laboratorio de series de tiempo — Flujo vehicular en la estación Illia (2017–2020)

### Plantilla colaborativa para Google Colab · Modelo principal: SARIMA

**Asignatura:** Laboratorio de Series de Tiempo  
**Dataset:** `flujo-vehicular-2017_2021_illia.csv`  
**Equipo:** Marco (líder), Aldana, Brickzon, Vera y Poma  
**Estado:** Plantilla de trabajo; **no contiene entrenamiento ni resultados finales**.

> **Pregunta del laboratorio:** ¿Qué tan bien puede SARIMA pronosticar el número de pasos vehiculares diarios de la estación Illia considerando sus patrones semanales?

> **Decisión metodológica inicial (sujeta a validación empírica):** serie diaria de `cantidad_pasos` agregada por `fecha`; modelo SARIMA con período estacional semanal `s=7`. La selección está motivada por diagnósticos exploratorios previos, pero no presupone exactitud predictiva.

# Organización y reglas compartidas

| Integrante | Bloque a cargo | Subapartados | Alcance |
|---|---|---:|---|
| **Aldana** | Ingesta, calidad y preparación | 5 | CSV completo, inconsistencias, duplicados, serie y particiones |
| **Brickzon** | Exploración y visualización | 5 | Tendencias, ciclo semanal/horario, STL y cambios de nivel |
| **Vera** | Diagnóstico estadístico y diseño de parámetros | 5 | ADF, KPSS, ACF/PACF, diferencias, candidatos y controles |
| **Poma** | Modelado y evaluación | 5 | Baseline, entrenamiento, validación, prueba y residuos |
| **Marco (líder)** | Introducción, conclusiones e integración | 3 | Alcance, interpretación final y control de entrega |

*Reglas de integración del equipo:*

- *Trabajen en este mismo notebook de Colab y coordinen antes de reemplazar celdas ajenas. En el historial de versiones de Colab, anoten cambios relevantes por responsable.*
- *No cambien el archivo fuente; toda transformación se hace en memoria y debe ser reproducible ejecutando las celdas en orden desde un entorno nuevo.*
- *Cada subsección tiene cuatro celdas separadas: título, encargado, indicaciones en cursiva y espacio de desarrollo. No eliminen las tres primeras al implementar.*
- *No inventen métricas ni resultados. Si una prueba no puede ejecutarse, escriban la razón y el impacto en el análisis.*
- *Los ejemplos del profesor son referencias de aprendizaje, no código obligatorio. No añadan LSTM, BiLSTM ni Prophet: el alcance acordado es un único modelo principal SARIMA y un baseline sencillo para contextualizar el error.*
- *La exploración descriptiva puede mostrar toda la historia 2017–2020, pero las decisiones de parámetros, ajustes y selección del modelo deben basarse solo en el entrenamiento y la validación, nunca en el conjunto final de prueba.*
- *No hagan divisiones aleatorias de los datos de series temporales. Definan cortes cronológicos fijos, una única estrategia de pronóstico y un horizonte coherente para comparar métodos.*
- *Antes de considerar terminada una sección, añadan bajo los resultados una interpretación breve en sus propias palabras, indicando qué se observó y qué limitación tiene.*

**Contrato de variables para enlazar secciones**

| Variable | Tipo y significado | La produce |
|---|---|---|
| `df_raw` | `pandas.DataFrame` con los registros originales | Aldana |
| `serie_diaria` | `pandas.Series` numérica, índice diario completo, nombre `cantidad_pasos` | Aldana |
| `train`, `val`, `test` | Series cronológicas sin solapamiento; validación 56 días y prueba 28 días | Aldana |
| `candidatos_sarima` | Lista de parámetros `(order, seasonal_order)` fundamentados | Vera |
| `tabla_validacion` | Comparación de candidatos y baseline en validación | Poma |
| `modelo_final` | SARIMA reentrenado con `train + val` | Poma |
| `pred_test`, `pred_base_test` | Pronósticos con índice igual al de `test` | Poma |
| `tabla_metricas` | Resultados de MAE, RMSE y mejora frente al baseline | Poma |

*Si alguna variable no puede construirse con garantías de calidad, documenten el impedimento y acuerden con Marco el tratamiento antes de avanzar. El bloque de entrenamiento no debe ocultar problemas detectados en preparación.*

# 1. Planteamiento del laboratorio

## 1.1. Problema, objetivo y justificación inicial

**Encargado: Marco (líder)**

*Propósito: Definir en términos académicos qué se pronosticará y por qué se estudiará una serie temporal diaria con SARIMA.*

*Indicaciones de desarrollo:*
1. *Redacta un planteamiento breve del problema: pronóstico del total diario de pasos vehiculares en Illia, no predicción individual de vehículos, tipo de pago ni sentido.*
2. *Incluye un objetivo general y de dos a tres objetivos específicos medibles: construir la serie diaria, describir su comportamiento, diagnosticar estacionalidad semanal y evaluar pronósticos fuera de muestra.*
3. *Explica por qué SARIMA es el candidato inicial: existen patrones semanales, pero su rendimiento se comprobará con validación y prueba; no prometas que es el modelo más preciso.*
4. *Delimita el periodo a 2017–2020; el nombre del CSV menciona 2021, pero el diagnóstico anterior no encontró registros de ese año.*
5. *Explicita que `cantidad_pasos` representa un conteo agregado; evita equiparar sin explicación cada registro del CSV a un vehículo individual.*

*Entregable verificable: Un texto de 2–4 párrafos y objetivos verificables, listo para exposición.*

*Criterios de aceptación: No hay afirmaciones de causalidad ni de precisión no demostradas; el objetivo coincide con lo que realmente se evaluará.*

> **[Marco: redactar aquí la introducción, el objetivo general y los objetivos específicos.]**

**Problema:** [Completar]

**Objetivo general:** [Completar]

**Objetivos específicos:** [Completar]

**Justificación del modelo elegido:** [Completar]

# 2. Carga, auditoría y preparación de los datos — Aldana

## 2.1. Configuración de Colab y lectura segura del CSV

**Encargado: Aldana**

*Propósito: Preparar un entorno reproducible y cargar el archivo completo sin alterar los registros originales.*

*Indicaciones de desarrollo:*
1. *Confirma que el notebook se ejecuta en Google Colab con Python 3; utiliza las bibliotecas disponibles y anota las versiones relevantes de pandas, numpy y statsmodels. Instala solo lo indispensable.*
2. *Permite dos formas de ubicar el CSV: subirlo al panel Archivos de Colab (`/content`) o montarlo desde Google Drive; documenta en una línea qué ruta deben editar los compañeros.*
3. *Lee las ocho columnas con pandas preservando las cadenas originales; comprueba el delimitador y la lectura UTF-8 sin reinterpretar etiquetas dañadas a ciegas. Si la lectura falla, comunica el problema en la salida.*
4. *Muestra dimensiones, nombres de columnas y `head()`/`dtypes`; evita imprimir los 1.048.575 registros completos en Colab.*
5. *Mantén `df_raw` como referencia inmutable para controles posteriores; usa copias para transformar fechas o categorías.*

*Entregable verificable: Bloque de inicialización funcional y variable `df_raw`, con tamaño del archivo verificado.*

*Criterios de aceptación: Al reiniciar el entorno y corregir solo `RUTA_CSV`, la celda carga datos sin rutas locales de Windows ni secretos.*

In [ ]:
# TODO — ALDANA: configuración y carga real del CSV.
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Cambia esta ruta al cargar el archivo en Colab o al montarlo en Drive.
RUTA_CSV = Path('/content/flujo-vehicular-2017_2021_illia.csv')

# Parámetros compartidos: no alterar sin documentarlo con todo el equipo.
SEMILLA = 42
PERIODO_SEMANAL = 7
DIAS_VALIDACION = 56
DIAS_PRUEBA = 28

print(f'Python: {sys.version.split()[0]} | pandas: {pd.__version__} | numpy: {np.__version__}')
print(f'CSV configurado: {RUTA_CSV} | existe: {RUTA_CSV.exists()}')
# TODO: si usarás Drive, añadir drive.mount('/content/drive') y ajustar RUTA_CSV.
# TODO: leer CSV en df_raw, validar 8 columnas y mostrar una muestra segura.

## 2.2. Validación de esquema, fechas, horas y valores

**Encargado: Aldana**

*Propósito: Identificar de forma cuantitativa problemas en los tipos y campos imprescindibles para agregación temporal.*

*Indicaciones de desarrollo:*
1. *Comprueba la presencia y el orden lógico de las columnas: `periodo`, `fecha`, `hora_inicio`, `estacion`, `sentido`, `tipo_vehiculo`, `forma_pago` y `cantidad_pasos`.*
2. *Analiza nulos por columna, tipos recibidos, fechas no interpretables, horas fuera de 0–23, año de `periodo` incompatible con `fecha` y cantidades nulas, negativas o no numéricas.*
3. *Convierte las fechas en una copia de trabajo: contempla `DD/MM/YYYY` y textos con componente `00:00`; usa `dayfirst=True` y verifica los extremos de calendario y los fallos de parseo.*
4. *Revisa el rango real de los años y verifica explícitamente si hay observaciones de 2021, sin asumirlo por el nombre del archivo.*
5. *Resume todos los hallazgos en una tabla corta con cantidad afectada, ejemplo y tratamiento propuesto; no corrijas ni elimines registros sin justificación.*

*Entregable verificable: Tabla de integridad y una copia de trabajo con fecha normalizada, sin modificar `df_raw`.*

*Criterios de aceptación: La conversión de fecha es verificable, los registros problemáticos quedan identificados y no se crean fechas silenciosamente.*

In [ ]:
# TODO — ALDANA: esquema, calidad y conversión de fechas en copia.
# Entradas: df_raw.
# Salidas sugeridas: df_trabajo, tabla_calidad.
# Verificar: columnas, nulos, tipos, fecha, hora, año, valores negativos y cero.

## 2.3. Duplicados, categorías y posible truncamiento

**Encargado: Aldana**

*Propósito: Documentar anomalías de captura que podrían sesgar los totales de tránsito.*

*Indicaciones de desarrollo:*
1. *Cuantifica filas duplicadas exactas y duplicados por clave lógica (`fecha`, `hora_inicio`, `estacion`, `sentido`, `tipo_vehiculo`, `forma_pago`); distingue la repetición literal de una clave con cantidades distintas.*
2. *Calcula cuántos pasos aportan las repeticiones exactas y presenta solo como simulación la diferencia de total si se descartaran, manteniendo intactos los datos base.*
3. *Tabula las categorías de `tipo_vehiculo` y `forma_pago` por año; registra etiquetas con el carácter `�` y cambios de taxonomía que impiden hacer comparaciones categóricas directas.*
4. *Señala que el archivo contiene aparentemente 1.048.575 filas, cifra coincidente con el límite de filas de Excel menos cabecera: es una alerta de posible truncamiento, no una prueba.*
5. *Cierra con la decisión de tratamiento para el análisis principal: conservar registros salvo evidencia sólida que permita depurar; documenta qué incertidumbre permanece.*

*Entregable verificable: Una tabla de duplicados, otra de categorías por año y un párrafo de riesgos.*

*Criterios de aceptación: Nunca se ejecuta `drop_duplicates` sobre la serie principal sin justificación y la auditoría no confunde duplicados con vehículos.*

In [ ]:
# TODO — ALDANA: auditoría de duplicados y taxonomías.
# Entradas: df_raw / df_trabajo.
# Salidas sugeridas: resumen_duplicados, categorias_por_anio.
# Importante: contabilizar sensibilidad sin eliminar filas de la fuente.

## 2.4. Construcción y verificación de la serie diaria

**Encargado: Aldana**

*Propósito: Crear la serie de modelado con una observación diaria y un conteo agregado de pasos.*

*Indicaciones de desarrollo:*
1. *Agrupa por la fecha normalizada y suma `cantidad_pasos` de todos los sentidos, horas, tipos de vehículo y formas de pago para producir un único valor por día.*
2. *Ordena el índice cronológicamente; construye el calendario esperado y distingue fechas sin registro de días que realmente tuvieron cero pasos. No rellenes ausencias con cero por defecto.*
3. *Comprueba el número de fechas, los extremos, la suma global frente a `df_raw`, valores negativos y duplicación accidental del índice.*
4. *Asegura que el resultado sea un `pd.Series` llamado `serie_diaria`, numérico, con índice `DatetimeIndex` y frecuencia diaria cuando sea válido.*
5. *Incluye una tabla pequeña con los primeros y últimos días y una figura inicial, explicando que la agregación cambia la unidad de análisis de filas categóricas a conteos por día.*

*Entregable verificable: `serie_diaria` lista para ser utilizada por Brickzon, Vera y Poma, con pruebas de integridad.*

*Criterios de aceptación: La suma de la serie coincide con el total de la fuente, salvo diferencias explícitamente documentadas, y no hay fechas duplicadas en el índice.*

In [ ]:
# TODO — ALDANA: construir serie diaria sin transformar destructivamente.
# Entradas: df_trabajo.
# Salida obligatoria: serie_diaria (pd.Series, DatetimeIndex diario).
# Verificar: sum(serie_diaria) == sum(df_raw['cantidad_pasos']).

## 2.5. Incidentes de 2018 y partición cronológica sellada

**Encargado: Aldana**

*Propósito: Registrar los conteos sospechosos y entregar los conjuntos fijos para validación y prueba.*

*Indicaciones de desarrollo:*
1. *Inspecciona los totales de los días 29/11/2018, 30/11/2018 y 01/12/2018; registra además las horas ausentes identificadas en el diagnóstico. No confundas bajo tránsito con una falla de registro demostrada.*
2. *Mantén la serie íntegra como análisis principal; si se realiza sensibilidad frente a anomalías, crea una variante claramente etiquetada y separada, nunca una sustitución silenciosa.*
3. *Separa cronológicamente los últimos 28 días de `serie_diaria` como `test` y los 56 días inmediatamente anteriores como `val`; el resto debe ser `train`.*
4. *Verifica las fechas iniciales/finales, número de puntos, ausencia de solapamientos y continuidad entre tres conjuntos. Muestra un gráfico de la separación y los conteos por segmento.*
5. *Declara expresamente que `test` queda reservado y no se usará para elegir orden, transformar datos con estadísticas futuras ni ajustar hiperparámetros.*

*Entregable verificable: Tres variables compartidas (`train`, `val`, `test`) y un registro de las anomalías de 2018.*

*Criterios de aceptación: 56 días de validación y 28 de prueba; series sin superposición y con índices diarios ordenados.*

In [ ]:
# TODO — ALDANA: documentar anomalías y sellar los cortes.
# Entradas: serie_diaria.
# Salidas obligatorias: train, val, test.
# Convención: test = últimos 28 días, val = 56 días anteriores,
# train = todo lo anterior; nunca usar test para elegir parámetros.

# 3. Exploración y descripción temporal — Brickzon

## 3.1. Estadísticas y evolución del flujo vehicular

**Encargado: Brickzon**

*Propósito: Describir la magnitud y dinámica del tránsito a lo largo del periodo observado.*

*Indicaciones de desarrollo:*
1. *Presenta una tabla de media, mediana, desviación estándar, mínimo, máximo y percentiles de la serie diaria; indica las fechas de los extremos.*
2. *Grafica el flujo diario completo con ejes y unidades legibles; añade una media móvil de 7 o 28 días para visualizar cambios de nivel sin reemplazar los datos originales.*
3. *Calcula totales por año y promedios diarios por año y mes, con una tabla compacta que permita comparar años con distinto número de días.*
4. *Comenta con números el contraste entre 2017–2019 y 2020, sin afirmar que un evento externo sea causalmente responsable.*
5. *Deja claras las diferencias entre un conteo diario, un total mensual y un promedio de días; evita etiquetar todo como cantidad de vehículos individuales.*

*Entregable verificable: Una tabla descriptiva y una o dos gráficas claras, con interpretación.*

*Criterios de aceptación: Las fechas de ejes son correctas, las unidades están definidas y todas las cifras proceden de la serie creada por Aldana.*

In [ ]:
# TODO — BRICKZON: estadísticas y curva temporal.
# Entradas: serie_diaria, train, val, test.
# Salidas: resumen_descriptivo, figuras de serie y tendencia.
# Los gráficos históricos pueden mostrar todo el rango; no usar test para ajustar modelos.

## 3.2. Patrón de día de la semana y fines de semana

**Encargado: Brickzon**

*Propósito: Medir y comunicar la estacionalidad semanal relevante para SARIMA.*

*Indicaciones de desarrollo:*
1. *Agrupa los días por lunes–domingo y calcula promedio, mediana y dispersión; ordena las etiquetas de forma natural para evitar gráficos alfabéticos.*
2. *Compara días laborables frente a fines de semana y reporta diferencias absolutas y porcentuales, indicando sobre qué conjunto se calcularon.*
3. *Construye un gráfico de barras de promedios por día y, si aporta valor, una representación que compare los perfiles de 2017, 2018, 2019 y 2020.*
4. *Analiza si se mantiene la forma semanal pese a los cambios de nivel; diferencia similitud de perfiles de estabilidad en los volúmenes.*
5. *Relaciona lo observado con la hipótesis de período estacional `s=7`, pero deja la confirmación estadística a las pruebas de Vera.*

*Entregable verificable: Tabla lunes–domingo, una figura semanal y un comentario numérico de estabilidad.*

*Criterios de aceptación: No se confunden correlaciones de perfiles con precisión predictiva y la elección de período se argumenta con observaciones.*

In [ ]:
# TODO — BRICKZON: ciclo semanal.
# Entradas: serie_diaria / train.
# Salidas: tabla_dia_semana, figura semanal y comparación anual.
# Preparar la explicación de la estacionalidad s=7.

## 3.3. Patrones intradiarios como contexto, no como objetivo

**Encargado: Brickzon**

*Propósito: Aprovechar la información de `hora_inicio` para explicar por qué se observa variación de tránsito, sin cambiar la granularidad de modelado.*

*Indicaciones de desarrollo:*
1. *A partir de `df_trabajo`, suma los pasos por hora 0–23 para obtener el perfil horario medio; usa una tasa por hora observada cuando compares promedios.*
2. *Identifica horas punta y valle y contrasta, si es pertinente, días laborables frente a fines de semana.*
3. *Revisa la existencia de huecos horarios documentados por Aldana; no imputes horas faltantes con cero para obtener promedios.*
4. *Produce a lo sumo una figura de línea por hora y una tabla compacta de máximos y mínimos; no construyas un segundo pipeline predictivo horario.*
5. *Explica por qué esta exploración es descriptiva y por qué el objetivo operativo y la evaluación se mantienen diarios.*

*Entregable verificable: Una visualización horaria acompañada de su interpretación y sus limitaciones.*

*Criterios de aceptación: No se entrena un modelo horario ni se confunden sumas acumuladas con promedios por intervalo.*

In [ ]:
# TODO — BRICKZON: perfil horario contextual.
# Entradas: df_trabajo (fecha normalizada, hora_inicio y cantidad_pasos).
# Salidas: figura del perfil 0–23 y tabla de horas altas/bajas.
# No cambiar la variable objetivo: seguirá siendo diaria.

## 3.4. Descomposición STL de la estacionalidad semanal

**Encargado: Brickzon**

*Propósito: Separar descriptivamente tendencia, componente semanal y residuos observados.*

*Indicaciones de desarrollo:*
1. *Aplica STL con período 7 a la serie de entrenamiento para no orientar decisiones metodológicas con el conjunto de prueba; informa si usas la opción robusta.*
2. *Presenta los componentes observada, tendencia, estacional y residual, manteniendo el calendario y escalas legibles.*
3. *Comenta la amplitud del patrón semanal, sus variaciones a través del tiempo y el papel de anomalías que afecten componentes.*
4. *Si reportas razones de varianzas de componentes, explica que no necesariamente suman 100 % porque pueden existir covarianzas.*
5. *Relaciona el resultado con el análisis por día de la semana y documenta si STL no resulta interpretable por cambios de régimen.*

*Entregable verificable: Gráfico de descomposición semanal y explicación de máximo dos párrafos.*

*Criterios de aceptación: El período STL es 7 y el diagnóstico de selección se hace con `train`, nunca con `test`.*

In [ ]:
# TODO — BRICKZON: STL robusto semanal con entrenamiento.
# Entradas: train.
# Salidas sugeridas: resultado_stl, figura de cuatro componentes.
# Usar statsmodels.tsa.seasonal.STL con period=7.

## 3.5. Cambios de nivel en 2020 y síntesis visual

**Encargado: Brickzon**

*Propósito: Contextualizar las rupturas de serie que podrían dificultar los pronósticos.*

*Indicaciones de desarrollo:*
1. *Compara nivel y dispersión mensual entre años, prestando atención a marzo y abril de 2020; presenta cifras, no solo impresiones visuales.*
2. *Señala el mínimo extraordinario de noviembre–diciembre de 2018 que Aldana encontró y explica cómo podría influir en líneas y promedios.*
3. *Usa una línea o gráfico mensual con anotaciones discretas para mostrar 2020; no declares automáticamente causas externas de cambios observados.*
4. *Escribe una conclusión de exploración con tres hallazgos cuantitativos que justifiquen estudiar un ciclo semanal y vigilar los cambios de régimen.*
5. *Entrega a Vera y Poma una nota de riesgos concretos (ruptura 2020 y registros incompletos 2018), sin proponer tratamientos no consensuados.*

*Entregable verificable: Una figura final y un resumen de hallazgos/limitaciones reutilizable en las conclusiones.*

*Criterios de aceptación: Todos los cambios se cuantifican y no se confunden hipótesis sobre causas con hechos.*

In [ ]:
# TODO — BRICKZON: análisis de 2020 y síntesis de figuras.
# Entradas: serie_diaria, train y tablas de Aldana.
# Salidas: comparación mensual/anual y hallazgos cuantitativos.

# 4. Pruebas estadísticas y plan de parametrización — Vera

## 4.1. Contrastes de estacionariedad ADF y KPSS

**Encargado: Vera**

*Propósito: Analizar formalmente la estabilidad estadística de la serie diaria de entrenamiento.*

*Indicaciones de desarrollo:*
1. *Ejecuta ADF sobre `train` en niveles y reporta estadístico, p-valor, rezagos y especificación de constante/tendencia; explica su hipótesis nula de raíz unitaria.*
2. *Ejecuta KPSS sobre la misma serie, reportando especificación y rezagos; recuerda que su hipótesis nula es estacionariedad, a diferencia de ADF.*
3. *Presenta los resultados de forma tabular y explica concordancias o contradicciones entre las pruebas con un nivel de significancia declarado (por ejemplo, 5 %).*
4. *Aclara que un cambio estructural en 2020 y la estacionalidad pueden influir en los contrastes, por lo que un p-valor no constituye una selección automática de SARIMA.*
5. *Si una prueba genera advertencias de límites de tabla o problemas numéricos, regístralos sin reemplazar valores reales por números inventados.*

*Entregable verificable: Tabla ADF/KPSS para entrenamiento en nivel y explicación de hipótesis.*

*Criterios de aceptación: Se especifica conjunto, parámetros de la prueba y conclusión condicionada, sin tocar `val` ni `test`.*

In [ ]:
# TODO — VERA: ADF/KPSS sobre train sin transformaciones futuras.
# Entradas: train.
# Salidas: tabla_estacionariedad_nivel.
# Bibliotecas: statsmodels.tsa.stattools.adfuller, kpss.

## 4.2. Autocorrelación y autocorrelación parcial

**Encargado: Vera**

*Propósito: Examinar las dependencias temporales y los rezagos consistentes con un ciclo semanal.*

*Indicaciones de desarrollo:*
1. *Calcula y grafica ACF y PACF en el conjunto de entrenamiento, especificando el número de rezagos y el método utilizado.*
2. *Reporta ACF de los rezagos 1, 7, 14 y 21 días; si la serie es no estacionaria, aclara el efecto de tendencias y cambios de régimen sobre estos valores.*
3. *Repite el análisis para transformaciones diagnósticas justificadas (por ejemplo primera diferencia o diferencia de 7 días) y distingue claramente cada figura.*
4. *Busca persistencia estacional en múltiplos de 7 sin asegurar que un pico determine inequívocamente los órdenes P, D y Q.*
5. *Explica en pocas líneas qué señales son útiles para acotar candidatos SARIMA y cuáles son ambiguas.*

*Entregable verificable: Figuras ACF/PACF y tabla breve de rezagos interpretados.*

*Criterios de aceptación: Los parámetros diagnósticos se obtienen solo de `train` y no se interpretan niveles no estacionarios como una prueba definitiva.*

In [ ]:
# TODO — VERA: ACF y PACF con train y transformaciones justificadas.
# Entradas: train.
# Salidas: figuras ACF/PACF y tabla_rezagos.
# Bibliotecas: statsmodels.graphics.tsaplots.plot_acf, plot_pacf.

## 4.3. Diferenciación regular y estacional

**Encargado: Vera**

*Propósito: Comprobar qué transformaciones estabilizan la serie antes de proponer parámetros `d` y `D`.*

*Indicaciones de desarrollo:*
1. *Construye sin alterar `train` una primera diferencia y una diferencia semanal de 7 días, registrando cuántas observaciones iniciales se pierden.*
2. *Repite ADF y KPSS cuando sean aplicables para series transformadas y presenta la comparación con la serie original.*
3. *Comprueba si la diferenciación excesiva introduce autocorrelación negativa fuerte o patrones residuales artificiales; no interpretes simplemente un p-valor pequeño como garantía de buen pronóstico.*
4. *Considera qué implica usar a la vez diferencia regular y estacional en términos de longitud e información preservada.*
5. *Recomienda únicamente rangos acotados de `d` y `D` para los candidatos, sin escoger hiperparámetros basados en la prueba final.*

*Entregable verificable: Tabla diagnóstica nivel / diferencia regular / diferencia semanal y sugerencias fundamentadas para `d` y `D`.*

*Criterios de aceptación: Se conoce cuántas filas pierde cada transformación; no se mezclan transformaciones calculadas con datos futuros.*

In [ ]:
# TODO — VERA: transformaciones diagnósticas sobre train.
# Entradas: train.
# Salidas: tabla_diferenciaciones y rangos de d, D.
# No sustituir ni reescalar train globalmente sin registrar decisiones.

## 4.4. Lista acotada de configuraciones SARIMA

**Encargado: Vera**

*Propósito: Entregar a Poma un espacio de búsqueda pequeño, explicable y viable en Colab.*

*Indicaciones de desarrollo:*
1. *Fija el período estacional `s=7` y prepara una lista de entre cuatro y ocho tuplas `(order, seasonal_order)` con órdenes enteros pequeños.*
2. *Justifica los rangos p/q y P/Q de acuerdo con ACF/PACF, diagnósticos de diferenciación y límites computacionales; no conviertas patrones gráficos en certezas.*
3. *Incluye al menos una configuración parsimoniosa como referencia, y evita búsquedas masivas o combinaciones que tarden demasiado sin añadir evidencia.*
4. *No ajustes SARIMA en esta sección: el entrenamiento y comparación de candidatos corresponden a Poma.*
5. *Registra por qué descartaste combinaciones demasiado complejas y acuerda una regla fija de selección por MAE de validación antes de ver `test`.*

*Entregable verificable: Variable `candidatos_sarima` y justificación escrita de cada familia de órdenes.*

*Criterios de aceptación: Toda tupla tiene período 7, la lista está acotada y no contiene decisiones basadas en test.*

In [ ]:
# TODO — VERA: entregar candidatos fundamentados a Poma.
# Formato obligatorio: [(order, seasonal_order), ...]
# Ejemplo de estructura (NO propuesta validada):
# candidatos_sarima = [((p, d, q), (P, D, Q, 7)), ...]
# No completar hasta revisar las pruebas sobre train.

## 4.5. Protocolo de validación y controles contra fuga temporal

**Encargado: Vera**

*Propósito: Establecer reglas verificables para comparar candidatos y proteger el conjunto de prueba.*

*Indicaciones de desarrollo:*
1. *Comprueba junto con Aldana los tres segmentos: `train` histórico, `val` de 56 días y `test` de 28 días, con fechas ordenadas y sin intersecciones.*
2. *Deja escrito que cada candidato se ajustará únicamente a `train` y generará un pronóstico directo de 56 días para `val`, sin utilizar observaciones de validación como datos ya conocidos dentro del horizonte.*
3. *Define que el ganador será el SARIMA válido con menor MAE de validación; el RMSE puede servir como desempate y la convergencia del optimizador debe registrarse.*
4. *Indica que, tras elegir el modelo, se reentrenará sobre `train + val` y se producirá una única predicción directa de los 28 días de `test`, con el mismo procedimiento para el baseline.*
5. *Propón controles contra fuga: no ajustar escaladores con la serie completa, no cambiar parámetros después de ver test, no reemplazar faltantes con estadísticas futuras y mantener índices alineados.*

*Entregable verificable: Protocolo escrito de validación y una celda de aserciones o verificaciones de fechas.*

*Criterios de aceptación: Poma puede implementarlo sin interpretar decisiones ambiguas; la prueba permanece independiente.*

In [ ]:
# TODO — VERA: validaciones de integridad temporal.
# Entradas: train, val, test, candidatos_sarima.
# Salidas: chequeos programáticos y protocolo documentado.
# Comprobar orden, continuidad, len(val)==56, len(test)==28 y ausencia de solape.

# 5. Entrenamiento, pronóstico y evaluación — Poma

## 5.1. Pronóstico base ingenuo estacional

**Encargado: Poma**

*Propósito: Construir una referencia simple para interpretar si SARIMA añade valor real.*

*Indicaciones de desarrollo:*
1. *Implementa un pronóstico semanal ingenuo que repite los últimos siete valores observados del segmento de entrenamiento para cubrir el horizonte futuro completo.*
2. *Para `val`, usa únicamente los últimos siete días de `train` y repítelos hasta completar 56 días; no uses datos reales de `val` dentro del mismo pronóstico directo.*
3. *Calcula MAE y RMSE del baseline en validación y alinea el índice pronosticado con `val`.*
4. *Después de la selección del modelo, vuelve a construir el baseline de prueba a partir de los siete últimos días de `train + val`, cubriendo los 28 días de test.*
5. *Documenta por qué existe este baseline: no es otro modelo principal ni requiere aprendizaje de parámetros, sino un mínimo de referencia para el valor del SARIMA.*

*Entregable verificable: Pronósticos base de validación y prueba, junto con métricas reproducibles.*

*Criterios de aceptación: No hay uso anticipado de observaciones reales del horizonte y ambas predicciones usan el mismo calendario que los valores verdaderos.*

In [ ]:
# POMA — 5.1. Baseline ingenuo estacional semanal; 56 días de validación.
# Entrada: particiones OFICIALES de Aldana. Sin transformación del CSV.
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.metrics import mean_absolute_error, mean_squared_error

for nombre in ('train', 'val', 'test'):
    if nombre not in globals():
        raise RuntimeError(f'Falta {nombre}: Aldana debe terminar 2.4–2.5. No se crean particiones artificiales.')

def _poma_verificar_segmento(s, nombre):
    if not isinstance(s, pd.Series) or not isinstance(s.index, pd.DatetimeIndex):
        raise TypeError(f'{nombre} debe ser pd.Series con DatetimeIndex.')
    if s.empty or s.index.has_duplicates or not s.index.is_monotonic_increasing:
        raise ValueError(f'{nombre}: fechas vacías, duplicadas o desordenadas.')
    if not pd.api.types.is_numeric_dtype(s) or s.isna().any():
        raise ValueError(f'{nombre}: valores nulos o no numéricos; consultar a Aldana.')
    if (s < 0).any() or not np.isfinite(s.to_numpy(dtype=float)).all():
        raise ValueError(f'{nombre}: conteos negativos o infinitos.')
    esperado = pd.date_range(s.index[0], periods=len(s), freq='D')
    if not esperado.equals(s.index):
        raise ValueError(f'{nombre}: días no consecutivos; no rellenar ausencias silenciosamente.')

for nombre in ('train', 'val', 'test'):
    _poma_verificar_segmento(globals()[nombre], nombre)
if len(train) < 14 or len(val) != 56 or len(test) != 28:
    raise ValueError('Se requiere train >= 14 días, val = 56 y test = 28.')
if train.index[-1] + pd.Timedelta(days=1) != val.index[0]:
    raise ValueError('train y val deben ser cronológicamente contiguos.')
if val.index[-1] + pd.Timedelta(days=1) != test.index[0]:
    raise ValueError('val y test deben ser cronológicamente contiguos.')

def _poma_baseline_semanal(historia, indice_destino):
    # Pronóstico DIRECTO: ningún dato del horizonte es incorporado mientras se pronostica.
    if len(historia) < 7:
        raise ValueError('Se requieren al menos siete observaciones históricas.')
    patron = historia.iloc[-7:].to_numpy(dtype=float)
    yhat = np.tile(patron, int(np.ceil(len(indice_destino) / 7)))[:len(indice_destino)]
    return pd.Series(yhat, index=indice_destino, name='baseline_s7')

def _poma_mae_rmse(real, pronostico):
    if not real.index.equals(pronostico.index):
        raise ValueError('Índices reales y pronosticados distintos.')
    if not np.isfinite(pronostico.to_numpy(dtype=float)).all():
        raise ValueError('Predicciones no finitas.')
    return (float(mean_absolute_error(real, pronostico)),
            float(np.sqrt(mean_squared_error(real, pronostico))))

pred_base_val = _poma_baseline_semanal(train, val.index)
mae_base_val, rmse_base_val = _poma_mae_rmse(val, pred_base_val)
metricas_base_val = {'MAE': mae_base_val, 'RMSE': rmse_base_val}
display(pd.DataFrame([{'Modelo':'Baseline s=7','MAE':mae_base_val,'RMSE':rmse_base_val}]))
display(Markdown(
    f'**Resumen automático (pendiente de revisión académica):** El baseline de repetición semanal obtuvo MAE={mae_base_val:,.2f} '
    f'y RMSE={rmse_base_val:,.2f} pasos diarios en 56 días de validación. '
    'Es solo una referencia de comparación; no presupone que la estacionalidad sea estable. '
    'El baseline de prueba se creará después de la selección en 5.3.'
))


## 5.2. Entrenamiento de candidatos SARIMA y validación

**Encargado: Poma**

*Propósito: Comparar las configuraciones predefinidas por Vera en un periodo de validación cronológico.*

*Indicaciones de desarrollo:*
1. *Usa `statsmodels` y ajusta cada configuración de `candidatos_sarima` exclusivamente sobre `train`, indicando explícitamente `(p,d,q)` y `(P,D,Q,7)`.*
2. *Para cada ajuste, genera un pronóstico directo de 56 pasos diarios hasta el final de `val`; alinea índices antes de medir errores.*
3. *Registra MAE, RMSE, tiempo de ejecución aproximado, advertencias relevantes y convergencia; captura errores por candidato sin ocultarlos.*
4. *Selecciona por menor MAE de validación entre ajustes válidos, usando la regla de desempate acordada; compara el resultado del ganador con el baseline de 5.1.*
5. *Si ningún candidato converge, informa a Vera y Marco antes de modificar el espacio de búsqueda; no explores `test` para encontrar un modelo más favorable.*

*Entregable verificable: `tabla_validacion` con todos los candidatos y los parámetros del mejor ajuste seleccionado por validación.*

*Criterios de aceptación: Cada fila puede reproducirse, el horizonte es el mismo y no se consulta el conjunto de prueba durante la selección.*

In [ ]:
# POMA — 5.2. SARIMA: entrenar SOLO con train; pronóstico directo 56 días.
# Entrada: candidatos_sarima justificados por Vera (4.4). Test permanece sellado.
import time
import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX

if 'candidatos_sarima' not in globals():
    raise RuntimeError('Falta candidatos_sarima: Vera debe completar 4.4.')
if not isinstance(candidatos_sarima, (list, tuple)) or not 4 <= len(candidatos_sarima) <= 8:
    raise ValueError('Vera debe entregar de 4 a 8 configuraciones SARIMA.')
if len(set(map(str, candidatos_sarima))) != len(candidatos_sarima):
    raise ValueError('Hay configuraciones repetidas.')

filas = []
for i, candidato in enumerate(candidatos_sarima, 1):
    t0 = time.perf_counter()
    capturadas = []
    avisos = []
    fila = {'Modelo':f'SARIMA_{i}', 'order':None, 'seasonal_order':None,
            'MAE':np.nan, 'RMSE':np.nan, 'loglikelihood':np.nan,
            'pronosticos_negativos':0, 'valor_minimo_pronosticado':np.nan,
            'segundos':np.nan, 'convergencia':False,
            'advertencias':'', 'estado':'ERROR'}
    try:
        if not isinstance(candidato, (tuple, list)) or len(candidato) != 2:
            raise ValueError('Formato inválido: ((p,d,q),(P,D,Q,7)).')
        order, seasonal_order = tuple(candidato[0]), tuple(candidato[1])
        if len(order) != 3 or len(seasonal_order) != 4:
            raise ValueError('Longitud incorrecta de órdenes.')
        if not all(isinstance(x, (int, np.integer)) and x >= 0 for x in order + seasonal_order[:3]):
            raise ValueError('Todos los órdenes deben ser enteros no negativos.')
        if seasonal_order[3] != 7:
            raise ValueError('La estacionalidad acordada es de siete días.')
        fila['order'], fila['seasonal_order'] = order, seasonal_order
        with warnings.catch_warnings(record=True) as capturadas:
            warnings.simplefilter('always')
            ajuste = SARIMAX(train, order=order, seasonal_order=seasonal_order,
                             enforce_stationarity=False,
                             enforce_invertibility=False).fit(disp=False, maxiter=200)
            fila['convergencia'] = bool(ajuste.mle_retvals.get('converged', False))
            parametros = np.asarray(ajuste.params, dtype=float)
            loglikelihood = float(ajuste.llf)
            fila['loglikelihood'] = loglikelihood
            if not np.isfinite(parametros).all():
                raise ValueError('Parámetros estimados no finitos.')
            if not np.isfinite(loglikelihood):
                raise ValueError('Log-verosimilitud no finita.')
            valores = np.asarray(ajuste.get_forecast(steps=56).predicted_mean, dtype=float)
        avisos.extend(f'{w.category.__name__}: {w.message}' for w in capturadas)
        if len(valores) != 56 or not np.isfinite(valores).all():
            raise ValueError('Pronóstico de validación inválido: se esperaban 56 valores finitos.')
        fila['pronosticos_negativos'] = int(np.count_nonzero(valores < 0))
        fila['valor_minimo_pronosticado'] = float(valores.min())
        if fila['pronosticos_negativos']:
            avisos.append(
                f'{fila["pronosticos_negativos"]} pronóstico(s) negativo(s); '
                f'mínimo={fila["valor_minimo_pronosticado"]:.6g}; no se recortan.'
            )
        candidato_val = pd.Series(valores, index=val.index)
        fila['MAE'], fila['RMSE'] = _poma_mae_rmse(val, candidato_val)
        fila['estado'] = 'VÁLIDO' if fila['convergencia'] else 'NO CONVERGIÓ'
    except Exception as error:
        avisos.append(f'{type(error).__name__}: {error}')
        fila['estado'] = 'ERROR'
    finally:
        # Las advertencias de ajuste y pronóstico se conservan aunque surja un error.
        avisos.extend(f'{w.category.__name__}: {w.message}' for w in capturadas
                      if f'{w.category.__name__}: {w.message}' not in avisos)
        fila['segundos'] = round(time.perf_counter()-t0, 2)
        fila['advertencias'] = ' | '.join(avisos)[:1200] or 'Ninguna'
        filas.append(fila)

base = {'Modelo':'Baseline s=7','order':None,'seasonal_order':None,
        'MAE':mae_base_val,'RMSE':rmse_base_val,'loglikelihood':np.nan,
        'pronosticos_negativos':0,'valor_minimo_pronosticado':np.nan,
        'segundos':0.0,'convergencia':None,'advertencias':'No aplica','estado':'REFERENCIA'}
tabla_validacion = pd.DataFrame([base] + filas)
display(tabla_validacion)
validos = [x for x in filas if x['estado']=='VÁLIDO' and
           np.isfinite(x['MAE']) and np.isfinite(x['RMSE']) and
           np.isfinite(x['loglikelihood'])]
if not validos:
    raise RuntimeError('Ningún SARIMA convergió con parámetros y log-verosimilitud finitos. No mirar test.')
ganador = min(validos, key=lambda x: (x['MAE'], x['RMSE'], x['Modelo']))
parametros_ganadores = ganador['order'], ganador['seasonal_order']
balance_val = 'menor' if ganador['MAE'] < mae_base_val else 'mayor o igual'
display(Markdown(
    f'**Resumen automático (pendiente de revisión académica):** El candidato elegido por MAE '
    f'mínimo (RMSE como desempate) fue {ganador["Modelo"]}, con MAE={ganador["MAE"]:,.2f} '
    f'y RMSE={ganador["RMSE"]:,.2f} en validación. Su MAE fue {balance_val} que el baseline '
    f'({mae_base_val:,.2f}). Solo participan ajustes convergentes con parámetros y '
    'log-verosimilitud finitos; test no intervino en la selección.'
))


## 5.3. Reentrenamiento final y pronóstico de 28 días

**Encargado: Poma**

*Propósito: Obtener el pronóstico fuera de muestra con la configuración ya elegida.*

*Indicaciones de desarrollo:*
1. *Congela la tupla ganadora y concatena `train + val` de forma cronológica; verifica que no haya índices repetidos.*
2. *Ajusta nuevamente SARIMA sobre toda esa historia disponible y genera un pronóstico directo de 28 días correspondientes exactamente a `test`.*
3. *Guarda el resultado como `pred_test` con índice de fecha igual al de `test`; conserva `modelo_final` y parámetros usados.*
4. *Obtén intervalos de confianza si el método los ofrece y reporta su nivel (por ejemplo 95 %); explica que sus supuestos pueden ser vulnerables a la ruptura de 2020.*
5. *No actualices el modelo usando días reales de `test` dentro del horizonte. Si el pronóstico produce valores negativos, registra el problema y justifica cualquier regla de posprocesamiento establecida antes de evaluar.*

*Entregable verificable: `modelo_final` y `pred_test` con sus 28 fechas, más intervalos si están disponibles.*

*Criterios de aceptación: El conjunto `test` se usa solo como verdad para evaluar, nunca como entrada anticipada al pronóstico.*

In [ ]:
# POMA — 5.3. Ajustar SARIMA ganador a train+val y pronosticar 28 días DIRECTOS.
if 'parametros_ganadores' not in globals():
    raise RuntimeError('Ejecutar 5.2: aún no hay parámetros seleccionados por validación.')
historia_final = pd.concat([train, val])
_poma_verificar_segmento(historia_final, 'historia_final')
if len(test) != 28:
    raise ValueError('La prueba debe contener exactamente 28 días.')
if historia_final.index[-1] + pd.Timedelta(days=1) != test.index[0]:
    raise ValueError('La historia y la prueba no son contiguas.')

inicio_final = time.perf_counter()
with warnings.catch_warnings(record=True) as ws_final:
    warnings.simplefilter('always')
    modelo_final = SARIMAX(historia_final, order=parametros_ganadores[0],
                           seasonal_order=parametros_ganadores[1],
                           enforce_stationarity=False,
                           enforce_invertibility=False).fit(disp=False, maxiter=200)
    objeto_pronostico = modelo_final.get_forecast(steps=28)
advertencias_finales = [f'{w.category.__name__}: {w.message}' for w in ws_final]
segundos_finales = round(time.perf_counter() - inicio_final, 2)
if not bool(modelo_final.mle_retvals.get('converged', False)):
    raise RuntimeError('El ajuste final no convergió: consultar a Vera y Marco sin mirar resultados de test.')

pronostico_array = np.asarray(objeto_pronostico.predicted_mean, dtype=float)
if len(pronostico_array) != 28 or not np.isfinite(pronostico_array).all():
    raise ValueError('Pronóstico final inválido.')
pred_test = pd.Series(pronostico_array, index=test.index, name='SARIMA')
# El baseline de test utiliza los últimos 7 días de train+val, no de test.
pred_base_test = _poma_baseline_semanal(historia_final, test.index)

# Los intervalos son opcionales, pero cualquier ausencia o anomalía queda explícita.
intervalos_test = None
estado_intervalos_test = ''
try:
    conf_int = np.asarray(objeto_pronostico.conf_int(alpha=0.05), dtype=float)
    if conf_int.shape != (28, 2):
        raise ValueError(f'dimensiones {conf_int.shape}; se esperaban (28, 2)')
    if not np.isfinite(conf_int).all():
        raise ValueError('los límites contienen valores no finitos')
    if (conf_int[:, 0] > conf_int[:, 1]).any():
        raise ValueError('hay límites inferiores mayores que los superiores')
    intervalos_test = pd.DataFrame(conf_int, index=test.index,
                                   columns=['inferior_95','superior_95'])
    estado_intervalos_test = 'Intervalos nominales del 95 % válidos en forma y orden.'
except Exception as error:
    estado_intervalos_test = (
        f'Intervalos de confianza no disponibles o inválidos: {type(error).__name__}: {error}'
    )
print('Orden seleccionado:', parametros_ganadores, '| tiempo (s):', segundos_finales)
print('Advertencias:', ' | '.join(advertencias_finales) or 'Ninguna')
print(estado_intervalos_test)
print('Índice de predicción:', pred_test.index.min(), '→', pred_test.index.max())
if (pred_test < 0).any():
    print('ADVERTENCIA: algunos valores pronosticados son negativos; no se recortan silenciosamente.')
display(pd.DataFrame({'SARIMA':pred_test, 'Baseline':pred_base_test}).head())
display(Markdown(
    '**Resumen automático (pendiente de revisión académica):** Se generó un pronóstico directo '
    'de 28 días sin incorporar observaciones reales de test al ajuste. Revisar las advertencias '
    'y el estado de los intervalos antes de redactar conclusiones; los intervalos nominales '
    'dependen de los supuestos del modelo.'
))


## 5.4. Métricas finales y visualización real frente a predicho

**Encargado: Poma**

*Propósito: Medir el error de predicción y explicar con claridad si SARIMA supera la referencia ingenua.*

*Indicaciones de desarrollo:*
1. *Calcula MAE y RMSE sobre las mismas 28 fechas para SARIMA y baseline; mantén unidades de pasos diarios.*
2. *Presenta una tabla con métricas de validación y una tabla con métricas finales de prueba, sin mezclarlas como si fueran comparables en los mismos datos.*
3. *Calcula la mejora porcentual frente al baseline con una fórmula explícita y explica si la diferencia es favorable, neutra o desfavorable.*
4. *Grafica los 28 días reales junto con el pronóstico SARIMA y el baseline; si dispones de intervalos, colócalos sin ocultar las curvas.*
5. *Describe un máximo y un mínimo de error en test, y qué significa su magnitud para la utilidad del pronóstico; evita afirmar que el modelo es superior a LSTM, Prophet o BiLSTM, porque no se entrenaron.*

*Entregable verificable: Variable `tabla_metricas`, gráfica de prueba y análisis conciso del desempeño.*

*Criterios de aceptación: Los índices reales/predichos coinciden, los cálculos no usan datos de entrenamiento como prueba y las fórmulas de métricas son correctas.*

In [ ]:
# POMA — 5.4. Validación y prueba SEPARADAS; MAE, RMSE y mejora vs baseline.
import matplotlib.pyplot as plt
for nombre in ('pred_test','pred_base_test','tabla_validacion'):
    if nombre not in globals():
        raise RuntimeError(f'Falta {nombre}: ejecutar los apartados anteriores.')
mae_sarima, rmse_sarima = _poma_mae_rmse(test, pred_test)
mae_base_test, rmse_base_test = _poma_mae_rmse(test, pred_base_test)

def _poma_mejora(ebase, emodelo):
    return float(100*(ebase-emodelo)/ebase) if ebase > 0 else np.nan

mejora_mae = _poma_mejora(mae_base_test, mae_sarima)
mejora_rmse = _poma_mejora(rmse_base_test, rmse_sarima)
tabla_metricas = pd.DataFrame([
    {'Modelo':'Baseline s=7','MAE':mae_base_test,'RMSE':rmse_base_test,
     'Mejora_MAE_pct':0.0,'Mejora_RMSE_pct':0.0},
    {'Modelo':'SARIMA','MAE':mae_sarima,'RMSE':rmse_sarima,
     'Mejora_MAE_pct':mejora_mae,'Mejora_RMSE_pct':mejora_rmse}
])
print('Métricas de VALIDACIÓN (56 días; selección):')
display(tabla_validacion[['Modelo','MAE','RMSE','estado']])
print('Métricas de PRUEBA (28 días; evaluación final):')
display(tabla_metricas)

fig, ax = plt.subplots(figsize=(12,5))
ax.plot(test.index, test.to_numpy(), 'o-', label='Real')
ax.plot(pred_test.index, pred_test.to_numpy(), 'o-', label='SARIMA')
ax.plot(pred_base_test.index, pred_base_test.to_numpy(), '--', label='Baseline semanal')
if intervalos_test is not None:
    ax.fill_between(test.index, intervalos_test['inferior_95'].to_numpy(),
                    intervalos_test['superior_95'].to_numpy(), alpha=.18,
                    label='Intervalo nominal 95 %')
ax.set(title='Pronóstico directo fuera de muestra: 28 días', xlabel='Fecha',
       ylabel='Pasos vehiculares diarios')
ax.legend(); ax.grid(alpha=.3); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

abs_error = (test-pred_test).abs()
fecha_max, fecha_min = abs_error.idxmax(), abs_error.idxmin()
resultado = ('superó' if mae_sarima < mae_base_test else
             'igualó' if np.isclose(mae_sarima, mae_base_test) else 'NO superó')
mejora_legible = f'{mejora_mae:+.2f} %' if np.isfinite(mejora_mae) else 'indefinida (MAE base=0)'
display(Markdown(
    f'**Resumen automático (pendiente de revisión académica):** SARIMA {resultado} al baseline en MAE de prueba. '
    f'SARIMA: MAE={mae_sarima:,.2f}, RMSE={rmse_sarima:,.2f}; '
    f'baseline: MAE={mae_base_test:,.2f}, RMSE={rmse_base_test:,.2f}. '
    f'Mejora relativa en MAE: {mejora_legible} '
    '(fórmula: 100×(MAE_base−MAE_SARIMA)/MAE_base). '
    f'Error absoluto máximo: {abs_error.max():,.2f} pasos ({fecha_max:%Y-%m-%d}); '
    f'mínimo: {abs_error.min():,.2f} pasos ({fecha_min:%Y-%m-%d}). '
    'La conclusión se limita a este único horizonte de prueba.'
))


## 5.5. Diagnóstico de residuos, sensibilidad y límites del modelo

**Encargado: Poma**

*Propósito: Evaluar críticamente errores y supuestos en lugar de concluir solo a partir de métricas.*

*Indicaciones de desarrollo:*
1. *Extrae residuos del SARIMA ajustado e inspecciona su evolución y autocorrelación; considera un contraste Ljung–Box con rezagos justificados, evitando afirmar independencia por un único p-valor.*
2. *Observa si quedan patrones semanales importantes y si la dispersión del error cambia en diferentes periodos de entrenamiento.*
3. *Discute si la ruptura de 2020, los registros anómalos de 2018 y la ausencia de covariables externas limitan la interpretación o las proyecciones.*
4. *Si el equipo realiza una sensibilidad a días sospechosos, declárala como análisis complementario separado de la evaluación principal y evita modificar la prueba para mejorar resultados.*
5. *Registra advertencias de convergencia, tiempo aproximado y condiciones donde el pronóstico podría fallar; entrega a Marco dos fortalezas y dos limitaciones cuantificadas cuando sea posible.*

*Entregable verificable: Gráficos/resumen de residuos y limitaciones fundamentadas para el cierre.*

*Criterios de aceptación: No se presenta una validación in-sample como si fuera prueba externa; los supuestos y problemas de calidad permanecen visibles.*

In [ ]:
# POMA — 5.5. Diagnóstico de residuos, Ljung–Box, dispersión y limitaciones.
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf
if 'modelo_final' not in globals() or 'tabla_metricas' not in globals():
    raise RuntimeError('Ejecutar 5.3–5.4 antes del diagnóstico.')

# Preferir innovaciones de un paso estandarizadas, que consideran la varianza predictiva.
# Si statsmodels no las expone de forma utilizable, usar residuos crudos y dejar constancia.
fuente_residuos = 'innovaciones estandarizadas de un paso'
try:
    innovaciones = np.asarray(
        modelo_final.filter_results.standardized_forecasts_error, dtype=float
    )
    if innovaciones.ndim == 2:
        if innovaciones.shape[0] != 1:
            raise ValueError('se esperaba una sola variable observada')
        innovaciones = innovaciones[0]
    if innovaciones.ndim != 1 or len(innovaciones) != len(historia_final):
        raise ValueError('dimensiones incompatibles con train+val')
    residuos_crudos = pd.Series(innovaciones, index=historia_final.index, name='innovacion_estandarizada')
except Exception as error_innovaciones:
    fuente_residuos = 'residuos crudos (fallback; no se obtuvieron innovaciones estandarizadas)'
    residuos_crudos = pd.Series(np.asarray(modelo_final.resid, dtype=float),
                                index=historia_final.index, name='residuo')
    print('Aviso sobre estandarización:', type(error_innovaciones).__name__, error_innovaciones)

residuos_crudos = residuos_crudos.replace([np.inf, -np.inf], np.nan)
# Omitir solo la inicialización difusa/kalmánica; reportar cuántos puntos se excluyen.
descartar = min(max(0, int(getattr(modelo_final, 'loglikelihood_burn', 0))), len(residuos_crudos))
residuos = residuos_crudos.iloc[descartar:].dropna()
print('Fuente de residuos:', fuente_residuos)
print('Residuos omitidos por inicialización o no finitos:', len(residuos_crudos) - len(residuos))
if len(residuos) < 30:
    raise ValueError('Cantidad insuficiente de residuos utilizables para diagnóstico (mínimo 30).')

# Para SARIMA, los grados de libertad consumidos se aproximan por p+q+P+Q.
# Cada rezago Ljung–Box usado debe dejar lag - model_df > 0.
orden, orden_estacional = parametros_ganadores
model_df_ljung_box = int(orden[0] + orden[2] + orden_estacional[0] + orden_estacional[2])
rezagos_propuestos = [7, 14, 21]
rezagos = [lag for lag in rezagos_propuestos
           if lag > model_df_ljung_box and lag < len(residuos)]
if not rezagos:
    raise ValueError(
        f'Ningún rezago propuesto {rezagos_propuestos} permite grados de libertad '
        f'positivos con model_df={model_df_ljung_box} y n={len(residuos)}. '
        'Revisar la especificación del diagnóstico antes de interpretar Ljung–Box.'
    )
if any(lag - model_df_ljung_box <= 0 for lag in rezagos):
    raise ValueError('Ljung–Box requiere lag - model_df > 0 para todos los rezagos.')
diagnostico_residuos = acorr_ljungbox(
    residuos, lags=rezagos, model_df=model_df_ljung_box, return_df=True
)
display(diagnostico_residuos)

fig, axes = plt.subplots(3, 1, figsize=(12,9))
axes[0].plot(residuos.index, residuos, linewidth=.8)
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set(title=f'Residuos: {fuente_residuos}', ylabel='Innovación / pasos')
axes[1].plot(residuos.index, residuos.rolling(28,min_periods=14).std())
axes[1].set(title='Dispersión móvil (28 días)', ylabel='Desv. estándar')
plot_acf(residuos, lags=min(28,len(residuos)-1), ax=axes[2], zero=False)
axes[2].set(title='ACF de residuos', xlabel='Rezago (días)')
for ax in axes: ax.grid(alpha=.25)
plt.tight_layout(); plt.show()

lag_referencia = rezagos[0]
p_referencia = float(diagnostico_residuos.loc[lag_referencia, 'lb_pvalue'])
if p_referencia < .05:
    lectura = (f'Para Ljung–Box al rezago {lag_referencia}, se rechaza la hipótesis nula '
               'de ausencia de autocorrelación hasta ese rezago al nivel 5 %.')
else:
    lectura = (f'Para Ljung–Box al rezago {lag_referencia}, no se rechaza la hipótesis nula '
               'de ausencia de autocorrelación al nivel 5 %; esto no demuestra independencia '
               'ni ausencia de estructura en todos los rezagos.')
limitaciones = [
    'cambios de nivel de 2020, que pueden deteriorar el pronóstico',
    'potenciales anomalías de registro de 2018, pendientes de contextualización por Aldana',
    'ausencia de covariables externas (feriados, clima, etc.)',
    'una única partición de prueba de 28 días, que no cubre todos los regímenes'
]
print('Grados de libertad consumidos (p+q+P+Q):', model_df_ljung_box)
print('Rezagos evaluados:', rezagos, '| grados de libertad residuales:',
      [lag-model_df_ljung_box for lag in rezagos])
print('Advertencias ajuste final:', ' | '.join(advertencias_finales) or 'Ninguna')
print('Tiempo ajuste final (s):', segundos_finales)
display(Markdown(
    '**Resumen automático (pendiente de revisión académica):** ' + lectura +
    ' El contraste utiliza grados de libertad ajustados por términos AR y MA regulares y '
    'estacionales. Un resultado no significativo no prueba independencia; revisar también '
    'la ACF y la dispersión. **Limitaciones a discutir:** ' + '; '.join(limitaciones) +
    '. Los residuos corresponden a train+val y no sustituyen la prueba externa de 5.4.'
))
# No realizar análisis de sensibilidad ni modificar el modelo o test en esta celda.


# 6. Conclusiones y entrega — Marco

## 6.1. Resultados, conclusiones y respuesta a la pregunta

**Encargado: Marco (líder)**

*Propósito: Sintetizar el trabajo sin rehacer el código estadístico o predictivo de los compañeros.*

*Indicaciones de desarrollo:*
1. *Redacta un párrafo que recuerde el problema, la unidad (pasos por día), la cobertura real 2017–2020 y por qué se propuso SARIMA estacional semanal.*
2. *Resume en lenguaje sencillo dos hallazgos exploratorios sustentados por las figuras y un hallazgo estadístico de Vera.*
3. *Usa los valores reales de MAE/RMSE y la comparación contra el baseline de Poma para explicar si la predicción fue útil; si SARIMA no supera al baseline, dilo expresamente.*
4. *Expón al menos dos limitaciones: cambio de nivel en 2020, anomalías de cobertura 2018, definición y duplicados no aclarados, o ausencia de variables externas.*
5. *Cierra contestando la pregunta inicial con una conclusión condicionada a los datos y al horizonte de prueba, sin afirmar superioridad frente a modelos que no se evaluaron.*

*Entregable verificable: Dos o tres párrafos finales basados en resultados de los cuatro compañeros.*

*Criterios de aceptación: Todas las cifras se pueden localizar en una celda anterior y ninguna afirmación contradice los resultados observados.*

> **[Marco: completar al disponer de las métricas y las figuras definitivas.]**

**Conclusión principal:** [Completar]

**Evidencias que la respaldan:** [Completar]

**Limitaciones y posibles mejoras:** [Completar]

## 6.2. Control final y preparación para la exposición

**Encargado: Marco (líder)**

*Propósito: Hacer una revisión breve de integración para que el archivo funcione como entrega de Colab.*

*Indicaciones de desarrollo:*
1. *Solicita a cada integrante revisar su bloque y confirmar que aparecen el título, la celda independiente de encargado, la celda de indicaciones en cursiva y sus resultados.*
2. *Comprueba el orden de ejecución tras reiniciar el entorno de Colab; si requiere un archivo externo, la ruta del CSV debe ser explícita y no quedar asociada a una computadora local.*
3. *Revisa que todas las gráficas tengan títulos, ejes y unidades, y que las tablas indiquen si son de entrenamiento, validación o prueba.*
4. *Confirma que las cifras de introducción, exploración y conclusión coincidan, que no existan resultados inventados y que las advertencias sobre calidad estén presentes.*
5. *Antes de entregar, guarda en Drive una copia con nombre de versión, asigna permisos adecuados a los integrantes y deja visibles las celdas de código/resultados necesarias para la sustentación.*

*Entregable verificable: Checklist final marcado y un notebook reproducible, consistente y apto para entrega.*

*Criterios de aceptación: No quedan TODO técnicos sin resolver; se ejecuta en orden y las decisiones metodológicas están explicadas.*

### Lista de verificación de entrega

- [ ] Aldana: CSV cargado, auditoría y serie diaria comprobadas.
- [ ] Brickzon: visualizaciones e interpretaciones revisadas.
- [ ] Vera: pruebas y candidatos documentados; no hay fuga temporal.
- [ ] Poma: validación, prueba, baseline y residuos listos.
- [ ] Marco: objetivos y conclusiones coherentes con los resultados.
- [ ] Todas las subsecciones conservan título, encargado e instrucciones en cursiva en celdas distintas.
- [ ] El notebook se ejecutó en Colab desde un entorno reiniciado.
- [ ] Se confirmó el acceso al CSV sin depender de rutas de Windows.
- [ ] Se guardó la versión final compartida del archivo.

---

### Nota de alcance para el equipo

*Esta es una plantilla de responsabilidades y criterios de aceptación. El diagnóstico previo registró una estacionalidad semanal, una fuerte caída de nivel en 2020, anomalías de conteo a fines de 2018 y posibles problemas de duplicación y truncamiento. Esas observaciones sirven para orientar la investigación, pero cada resultado que aparezca en el laboratorio debe surgir del análisis reproducible del CSV utilizado.*

*La elección de SARIMA fue una decisión previa de dirección académica. El objetivo de la validación es determinar su rendimiento real, no fabricar una conclusión favorable. El baseline estacional se incluye únicamente para contextualizar el desempeño de SARIMA, no como segundo modelo principal del laboratorio.*